# Visium HD and scANVI Analysis

**Approach.**
1. Reference: malignant cells from snRNA-seq atlas (cell states).
2. Query: tumour cells of the Visium HD object (published annotation).
3. Joint scVI model on shared genes (batch = dataset; covariate = sample), then scANVI with
   reference states as labels and query cells unlabelled.
4. Predicted state + confidence per query cell; low-confidence cells → `Uncertain`.
5. Write predictions back to the **full** spatial object as `cell_label_final`.

**Output.** `adata_sp_final_relabelled.h5ad` (all spatial cells, `cell_label_final`, `predicted_label`,
`prediction_score`, scANVI latent for tumour cells), saved models, QC and spatial figures.

In [ ]:
# !pip install --quiet scanpy squidpy "scvi-tools[cuda]"
# %cd <project>/spatial_visiumHD

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import squidpy as sq
import torch

## Config

In [ ]:
# --- inputs / outputs ---
reference_path = "./../h5ad/tumour/pulmonary_epi_Lin/sub_Epi_Tumor_Alveolar_raw_scvi_combine_TN_FDG_res06_mod_new_states_join_scores.h5ad"

query_path = "adata1_adata2_preprocess_join.h5ad"           # public Visium HD, both slides joined and annotated

output_path = "adata_sp_final_relabelled.h5ad"
model_dir = "models"
figures_output_dir = "figures"
tables_output_dir = "tables"
seed = 0

for d in (model_dir, figures_output_dir, tables_output_dir):
    os.makedirs(d, exist_ok=True)
sc.settings.figdir = figures_output_dir
scvi.settings.seed = seed
print("GPU available:", torch.cuda.is_available(), "| scvi-tools", scvi.__version__)

## 1. Reference: malignant states from the snRNA-seq atlas

In [ ]:
# --- reference (snRNA-seq atlas) ---
label_key = "final_state"
reference_states = ["1_Epithelial baseline", "2_Differentiated epithelial", "3_Devp_stem-like",
                    "4_Inflammatory EMT", "5_Neurod_EMT-like", "6_Proliferative"]
reference_timepoint_col = "recist_v3"
reference_timepoints = ["1_treatment_naive", "3_on_treatment", "4_TKI_resistance"]
reference_sample_col = "sample"

# --- model ---
counts_layers = ["counts", "raw_counts"] # first layer found is used as raw counts
min_cells_per_gene = 10
n_latent = 30
scvi_max_epochs = 200
scanvi_max_epochs = 50
train_models = True                      # False -> load saved models from model_dir
scvi_model_name = "scvi_ref_spatial"
scanvi_model_name = "scanvi_cancer_states"
confidence_threshold = 0.5               # max class probability; chance level for 6 states is ~0.17

def use_raw_counts(adata, name):
    for layer in counts_layers:
        if layer in adata.layers:
            adata.X = adata.layers[layer].copy()
            break
    else:
        print(f"{name}: no counts layer found, using X")
    vals = adata.X[:1000].data if hasattr(adata.X, "data") else np.asarray(adata.X[:1000]).ravel()
    assert np.allclose(vals, np.round(vals)) and vals.min() >= 0, f"{name}: X must contain raw counts"


adata_ref = sc.read_h5ad(reference_path)
adata_ref = adata_ref[adata_ref.obs[label_key].isin(reference_states)
                      & adata_ref.obs[reference_timepoint_col].isin(reference_timepoints)].copy()
use_raw_counts(adata_ref, "reference")
pd.crosstab(adata_ref.obs[label_key], adata_ref.obs[reference_timepoint_col], margins=True)

## 2. Query: tumour cells of the Visium HD object

The full object is kept (`adata_sp`); only its tumour cells (`adata_q`) are used for label transfer.

In [ ]:
# --- query (Visium HD) ---
query_label_col = "cell_type_2"          # published annotation
query_tumour_label = "tumour_cell"
query_sample_col = "sample"              # tissue section, e.g. ER_pre / ER_post
query_group_col = "response_group"       # ER / SR / LR1 / LR2 (patient-level, used for plots only)
library_key = "slide"                    # spatial library for squidpy

adata_sp = sc.read_h5ad(query_path)
use_raw_counts(adata_sp, "spatial")
print(adata_sp.obs[query_label_col].value_counts())

adata_q = adata_sp[adata_sp.obs[query_label_col] == query_tumour_label].copy()
pd.crosstab(adata_q.obs[query_sample_col], adata_q.obs[query_group_col])

## 3. Shared genes and joint object

Genes are intersected first, filtered for coverage in each dataset, and intersected again.
`batch_sample` (dataset + sample) is the nuisance covariate within each dataset.

In [ ]:
genes = adata_ref.var_names.intersection(adata_q.var_names)
print(f"shared genes: {len(genes)}")
adata_ref, adata_q = adata_ref[:, genes].copy(), adata_q[:, genes].copy()
sc.pp.filter_genes(adata_ref, min_cells=min_cells_per_gene)
sc.pp.filter_genes(adata_q, min_cells=min_cells_per_gene)
genes = adata_ref.var_names.intersection(adata_q.var_names)
adata_ref, adata_q = adata_ref[:, genes].copy(), adata_q[:, genes].copy()
print(f"shared genes after coverage filter: {len(genes)}")

for a, name, sample_col, label in [(adata_ref, "reference", reference_sample_col, None),
                                   (adata_q, "spatial", query_sample_col, "Unknown")]:
    a.obs["dataset"] = name
    a.obs["batch_sample"] = name + "_" + a.obs[sample_col].astype(str)
    if label is not None:
        a.obs[label_key] = label

adata = sc.concat([adata_ref, adata_q], join="inner", label="source", keys=["reference", "spatial"], index_unique="-")
adata.obs = adata.obs[["dataset", "batch_sample", label_key]]
adata.obs[label_key] = adata.obs[label_key].astype("category")
adata.obs["dataset"].value_counts()

## 4. Init scVI → scANVI

- `batch_key="dataset"`: main correction, snRNA-seq vs Visium HD.
- `categorical_covariate_keys=["batch_sample"]`: sample-level effects within each dataset.
- scANVI is initialised from scVI with reference states as labels, query cells `Unknown`.

In [ ]:
scvi_path = os.path.join(model_dir, scvi_model_name)
scanvi_path = os.path.join(model_dir, scanvi_model_name)
scvi.model.SCVI.setup_anndata(adata, batch_key="dataset", labels_key=label_key,
                              categorical_covariate_keys=["batch_sample"])

if train_models:
    vae = scvi.model.SCVI(adata, n_latent=n_latent)
    vae.train(max_epochs=scvi_max_epochs, early_stopping=True)
    vae.save(scvi_path, overwrite=True)
    scanvi_model = scvi.model.SCANVI.from_scvi_model(vae, labels_key=label_key, unlabeled_category="Unknown")
    scanvi_model.train(max_epochs=scanvi_max_epochs, early_stopping=True)
    scanvi_model.save(scanvi_path, overwrite=True)
else:
    scanvi_model = scvi.model.SCANVI.load(scanvi_path, adata=adata)

## 5. Predictions and confidence

The reference confusion table is a sanity check only (training cells).

In [ ]:
adata.obsm["X_scANVI"] = scanvi_model.get_latent_representation()
probs = scanvi_model.predict(adata, soft=True)
probs.index = adata.obs_names
adata.obs["predicted_label"] = probs.idxmax(axis=1)
adata.obs["prediction_score"] = probs.max(axis=1)

is_q = adata.obs["dataset"] == "spatial"
ref = adata.obs[~is_q]
print(f"reference agreement (training cells): {(ref['predicted_label'] == ref[label_key].astype(str)).mean():.1%}")
pd.crosstab(ref[label_key], ref["predicted_label"], normalize="index").round(2).to_csv(
    f"{tables_output_dir}/reference_confusion.csv")

q = adata.obs[is_q].copy()
q.index = q.index.str.removesuffix("-spatial")
q["predicted_label_conf"] = q["predicted_label"].where(q["prediction_score"] >= confidence_threshold, "Uncertain")
print(f"query cells below {confidence_threshold}: {(q['predicted_label_conf'] == 'Uncertain').sum()} / {len(q)}")
q["predicted_label_conf"].value_counts()

## 6. Build the final spatial object

Tumour cells receive their predicted state (or `Uncertain`); all other cells keep the published
label. Colours are fixed for tumour states and assigned from `other_palette` to the rest.

In [ ]:
# --- plotting ---
tumour_colors = {
    "1_Epithelial baseline": "#8dd3c7", "2_Differentiated epithelial": "#bebada", "3_Devp_stem-like": "#80b1d3",
    "4_Inflammatory EMT": "#fccde5", "5_Neurod_EMT-like": "#bc80bd", "6_Proliferative": "#ffed6f",
    "Uncertain": "#d9d9d9",
}
other_palette = ["#E16A86", "#AA9000", "#00AA5A", "#00A6CA", "#B675E0", "#0804F5", "#EF9708", "#3B99B1",
                 "#E093C3", "#005682", "#FCC602", "#4A6FE3", "#4B201D", "#7F3C37", "#91B88B", "#5C5C5C"]
sample_order = ["ER_pre", "ER_post", "SR_pre", "SR_post", "LR1_pre", "LR1_post", "LR2_pre", "LR2_post"]
spatial_subsets = {                      # file suffix -> labels shown on the tissue
    "macs_Tc_and_tc_cell_states": ["Macrophage", "T cell"] + list(tumour_colors)[:5],
    "macs_and_tc_cell_states": ["Macrophage"] + list(tumour_colors)[:5],
}

assert q.index.isin(adata_sp.obs_names).all(), "predicted barcodes not found in the spatial object"
adata_sp.obs["predicted_label"] = q["predicted_label_conf"].reindex(adata_sp.obs_names)
adata_sp.obs["prediction_score"] = q["prediction_score"].reindex(adata_sp.obs_names)
latent = pd.DataFrame(adata.obsm["X_scANVI"][is_q.to_numpy()], index=q.index)
adata_sp.obsm["X_scANVI"] = latent.reindex(adata_sp.obs_names).to_numpy()   # NaN for non-tumour cells

final = adata_sp.obs[query_label_col].astype(str)
tumour = adata_sp.obs["predicted_label"].notna()
final[tumour] = adata_sp.obs.loc[tumour, "predicted_label"]
adata_sp.obs["cell_label_final"] = final.astype("category")

others = sorted(c for c in adata_sp.obs["cell_label_final"].cat.categories if c not in tumour_colors)
if len(others) > len(other_palette):
    print(f"WARNING: {len(others)} non-tumour labels but {len(other_palette)} colours; colours will repeat")
color_map = {**tumour_colors, **{c: other_palette[i % len(other_palette)] for i, c in enumerate(others)}}
adata_sp.uns["cell_label_final_colors"] = [color_map[c] for c in adata_sp.obs["cell_label_final"].cat.categories]

adata_sp.write_h5ad(output_path)
adata_sp.obs["cell_label_final"].value_counts()

## 7. QC and figures

In [ ]:
sc.pp.neighbors(adata, use_rep="X_scANVI", random_state=seed)
sc.tl.umap(adata, random_state=seed)
sc.pl.umap(adata, color=["dataset", label_key, "predicted_label"], wspace=0.4, save="_scanvi_label_transfer.pdf")

fig, ax = plt.subplots(figsize=(6, 3))
tum_obs = adata_sp.obs[adata_sp.obs["prediction_score"].notna()]
for s, grp in tum_obs.groupby(query_sample_col, observed=True)["prediction_score"]:
    ax.hist(grp, bins=np.linspace(0, 1, 41), histtype="step", label=s)
ax.axvline(confidence_threshold, ls="--", c="red")
ax.set_xlabel("scANVI prediction score")
ax.legend(fontsize=7, frameon=False)
plt.savefig(f"{figures_output_dir}/prediction_score_per_sample.pdf", bbox_inches="tight")
plt.show()

In [ ]:
sq.pl.spatial_scatter(adata_sp, color="cell_label_final", library_key=library_key, img=True, size=12,
                      save="spatial_cell_label_final.pdf")
sq.pl.spatial_scatter(adata_sp, color="prediction_score", library_key=library_key, img=False, size=15,
                      save="spatial_prediction_score.pdf")

for suffix, labels in spatial_subsets.items():
    sub = adata_sp[adata_sp.obs["cell_label_final"].isin(labels)].copy()
    sub.obs["cell_label_final"] = sub.obs["cell_label_final"].cat.remove_unused_categories()
    sub.uns["cell_label_final_colors"] = [color_map[c] for c in sub.obs["cell_label_final"].cat.categories]
    sq.pl.spatial_scatter(sub, color="cell_label_final", library_key=library_key, img=True, img_alpha=0.1,
                          size=18, save=f"spatial_{suffix}.pdf")

Tumour-state composition per tissue section (confident predictions only), grouped by
response group with pre/post sections side by side.

In [ ]:
states = [s for s in tumour_colors if s != "Uncertain"]
obs_t = adata_sp.obs[adata_sp.obs["cell_label_final"].isin(states)]

props = pd.crosstab(obs_t[query_sample_col], obs_t["cell_label_final"], normalize="index").mul(100)
props = props.reindex(columns=states, fill_value=0)
props = props.loc[[s for s in sample_order if s in props.index]]
props.round(2).to_csv(f"{tables_output_dir}/tumour_state_percent_per_sample.csv")

fig, ax = plt.subplots(figsize=(max(4, 0.9 * len(props)), 4))
props.plot.bar(stacked=True, ax=ax, width=0.75, edgecolor="none", color=[tumour_colors[s] for s in states])
groups = props.index.str.rsplit("_", n=1).str[0]           # ER_pre -> ER
starts = [i for i in range(len(groups)) if i == 0 or groups[i] != groups[i - 1]]
for i, start in enumerate(starts):
    end = starts[i + 1] if i + 1 < len(starts) else len(groups)
    ax.text((start + end - 1) / 2, 103, groups[start], ha="center", va="bottom", fontsize=11)
    if start > 0:
        ax.axvline(start - 0.5, color="grey", ls="--", lw=1)
ax.set_ylim(0, 108)
ax.set_ylabel("Tumour cells (%)")
ax.set_xlabel("")
ax.legend(title="Tumour state", bbox_to_anchor=(1.01, 1), loc="upper left", frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.savefig(f"{figures_output_dir}/tumour_state_composition_by_sample.pdf", dpi=300, bbox_inches="tight") #Figure 2H
plt.show()
props.round(1)